##### Copyright 2026 The codelab-jax Authors.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at
[apache.org/licenses/LICENSE-2.0](https://www.apache.org/licenses/LICENSE-2.0).

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS, WITHOUT
WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.

<table align="left"><td>
  <a target="_blank" href="https://colab.research.google.com/github/xbill9/codelab-jax/blob/main/notebooks/01_jax_tpu_mechanics.ipynb">
    <img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab
  </a>
</td><td>
  <a target="_blank" href="https://github.com/xbill9/codelab-jax/blob/main/notebooks/01_jax_tpu_mechanics.ipynb">
    <img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />View source on GitHub
  </a>
</td></table>

# JAX on TPU: donation, static shapes, and cached state

Three things decide whether a JAX program is fast on a TPU, and none of them
are the model. This notebook measures all three on the TPU runtime you are
connected to right now.

By the end you will be able to explain:

1. **Why `jit` recompiles** when your input shape changes by one token, and
   what padding to a bucket costs and does not cost.
2. **What `donate_argnums` does to a KV cache** — measured here at roughly
   **1.6x** on a real decode loop, the single largest speedup in the work this
   notebook is drawn from.
3. **How to carry mutable state through a pure function** with
   `lax.dynamic_update_slice`, and how to prove the cached path agrees with a
   full re-forward.

Everything runs on one chip with synthetic arrays. No checkpoint, no gated
model, no Hugging Face token. Notebook 02 does the real thing.

## Setup

SOURCE: adapt the device/version preamble from `~/tpu-jax/ports/gemma4/
jax_e_smoke_test.py`. Assert we are actually on a TPU and print the chip
generation, because every number below is meaningless on CPU.

In [ ]:
# TODO: import jax, print jax.__version__, jax.devices(), device_kind.
# Fail loudly with a pointer to Runtime > Change runtime type if not TPU.

## 1. Why your jitted function keeps recompiling

SOURCE: the 128-aligned bucket padding in `~/tpu-jax/ports/gemma4/
jax_e_model.py`, and the "padding to 128-aligned TPU buckets does not change
model output" claim verified in `~/tpu-jax/tests/test_chunked_prefill.py`.

Beats to hit:
- Trace a trivial function over sequence lengths 5, 6, 7 -> three compiles.
- Show the compile cost with `jax.block_until_ready` and a timer.
- Introduce bucketing to 128. Three lengths, one compile.
- Then the part tutorials skip: show the padded result is *bitwise identical*
  to the unpadded one, so bucketing is free correctness-wise and only costs
  wasted FLOPs.

In [ ]:
# TODO

## 2. Buffer donation

SOURCE: `~/tpu-jax/tests/test_donation.py` and section 1 of
`~/tpu-jax/benchmarks/runs/2026-07-29-kv-quant-v6e1/REPORT.md`.

The finding to reproduce: without `donate_argnums`, a single-token
`dynamic_update_slice` into a KV cache can leave *two* full caches live at
once. Donation removes the copy. Measured on v6e-1 at 1.60-1.62x faster with
a BF16 KV cache, and it nearly doubled the resident-token ceiling.

Beats to hit:
- Build a KV-cache-shaped array (layers x batch x heads x seq x dim).
- Time an update step without donation; watch peak HBM via
  `jax.local_devices()[0].memory_stats()`.
- Add `donate_argnums`, time again, show both the speedup and the drop in
  peak bytes.
- Show the donated-buffer-reuse error you get if you touch the input after,
  because that is the trap and readers will hit it.

CAVEAT for the reader: 1.6x is *this* shape on *this* chip. Have them read
their own number off the cell rather than quoting mine.

In [ ]:
# TODO

## 3. Cached decode, and proving it correct

SOURCE: `~/tpu-jax/tests/test_kv_cache_parity.py` — "cached decode matches
full-sequence re-forward within float32 tolerance".

Beats to hit:
- A toy attention block, run two ways: full re-forward over the whole prefix
  each step, vs. append-to-cache with `lax.dynamic_update_slice`.
- Assert `allclose`. This is the test that catches an off-by-one in the write
  index, which is the most common way a hand-rolled cache goes wrong.
- Plot tokens/s for both as sequence length grows: the re-forward curve is
  quadratic, the cached one flat.

In [ ]:
# TODO

## 4. The decode budget is a constant

SOURCE: section 1 of `2026-07-29-kv-quant-v6e1/REPORT.md` ("The decode budget
is a constant, to 0.0%") and section 2c of `2026-07-28-jax-e2b-v6e1/REPORT.md`
("we are memory-limited, not latency-limited").

The point: single-token decode moves the whole KV cache through memory every
step, so it is bandwidth-bound, not compute-bound. Derive bytes/token, divide
by the chip's HBM bandwidth, compare to the measured step time. When those
two agree you have understood the machine.

This is also the honest framing for why the int8 KV cache in the next section
helps: it is a bandwidth cut, not a math cut.

In [ ]:
# TODO

## 5. An int8 KV cache

SOURCE: `~/tpu-jax/tests/test_quantized_kv.py` and section 2 of
`2026-07-29-kv-quant-v6e1/REPORT.md` — 1.17-1.19x faster than donated BF16,
and 1.82-1.98x the capacity.

Beats to hit:
- Per-head scale, quantize on write, dequantize on read.
- Verify against a dequantize-first reference (that is the real test).
- Measure both axes: the speedup *and* the capacity, since capacity is the
  bigger win and the one people forget to measure.
- Quality: the source work measured 28.41 vs 28.73 perplexity and 97.08%
  greedy-token agreement over 583 steps. We cannot redo that here without a
  checkpoint -- state the number, cite notebook 02, do not re-derive it.

In [ ]:
# TODO

## 6. A note on multiple chips

Colab gives you one TPU chip (v5e-1 or v6e-1), so `jax.sharding` and
`shard_map` cannot be *measured* here. Rather than hand-wave, show the API on
emulated devices and label it as emulation:

    XLA_FLAGS=--xla_force_host_platform_device_count=8

Build a `Mesh`, shard an array, show the partition spec, and say plainly that
the numbers are CPU and the point is the API surface. Then point at the real
multi-chip path (v6e-8 on GCE or GKE).

DECISION: keep this section short, or drop it if a Marathon teammate takes
sharding as their own notebook. Do not fake a multi-chip result.

In [ ]:
# TODO

## What to take away

TODO: restate the three mechanics, and hand off to notebook 02, which applies
all of them to a real Gemma 4 checkpoint.